In [ ]:
from pathlib import Path

import numpy as np
from skimage.feature import multiscale_basic_features
from skimage.io import imread
from skimage.segmentation import relabel_sequential
from sklearn.ensemble import RandomForestClassifier
from scipy.ndimage import gaussian_gradient_magnitude

from snap_to_edge import snap_labels_to_edge

In [ ]:
image_path = '/Users/david/Desktop/weihua_hp1_cluster_annotations/images'
mask_path = '/Users/david/Desktop/weihua_hp1_cluster_annotations/masks'

do_snap_to_edge = True
snap_to_edge_radius = 2
snap_to_edge_ggm_sigma = 1.0

# how many pixels per class and image to sample
max_pixels_per_class = 100_000

# whether to sample pixels with replacement
# if True, we will use max_pixels_per_class for each class
sample_with_replacement = True

In [ ]:
# get image-mask pairs
image_files = sorted(Path(image_path).glob('*.tif'))
mask_files = sorted(Path(mask_path).glob('*.tif'))

# show for verification
list(zip(image_files, mask_files))

In [ ]:
train_x = []
train_y = []

for idx in range(len(image_files)):

    img = imread(image_files[idx]).astype(float)
    mask = imread(mask_files[idx]).astype(int)

    mask, _, _ = relabel_sequential(mask)

    # refine masks with snap-to-edge
    if do_snap_to_edge:
        mask_ref = snap_labels_to_edge(mask, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)
    else:
        mask_ref = mask

    # calculate multiscale features (similar to ilastik, etc.)
    features = multiscale_basic_features(img)

    # flatten mask and features
    features_flat = features.reshape((-1, features.shape[-1]))
    mask_flat = mask_ref.ravel()

    for label in np.unique(mask_flat):

        # pixels of class
        selection = np.flatnonzero(mask_flat == label)

        # sampled selection of those pixels
        if sample_with_replacement:
            selection_to_keep = np.random.choice(selection, max_pixels_per_class, replace=True)
        else:
            selection_to_keep = np.random.choice(selection, min(max_pixels_per_class, len(selection)), replace=False)

        # add to training x, y
        train_x.append(features_flat[selection_to_keep])
        train_y.append(mask_flat[selection_to_keep])

# concat into one dataset
train_x = np.concat(train_x)
train_y = np.concat(train_y)

# fit RF
model = RandomForestClassifier(n_jobs=-1)
model.fit(train_x, train_y)

In [ ]:
from skl2onnx import to_onnx

onx = to_onnx(model, train_x[:1])

with open("rf_hp1_try001.onnx", "wb") as f:
    f.write(onx.SerializeToString())

### Test snap-to-edge

Here, we load a single image + mask and show snap-to-edge results

In [ ]:
import napari

idx = 1

img = imread(image_files[idx]).astype(float)
mask = imread(mask_files[idx]).astype(int)

mask, _, _ = relabel_sequential(mask)

# refine mask with edge snap
mask_ref = snap_labels_to_edge(mask, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.view_image(img)
viewer.add_labels(mask)
viewer.add_labels(mask_ref)

### Test segmentation

In [ ]:
idx = 1

img = imread(image_files[idx]).astype(float)
mask = imread(mask_files[idx]).astype(int)

mask, _, _ = relabel_sequential(mask)

mask_ref = snap_labels_to_edge(mask, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), radius_morphology=snap_to_edge_radius)

features = multiscale_basic_features(img)
features_flat = features.reshape((-1, features.shape[-1]))


In [ ]:
# predict with sklearn model in memory
mask_pred = model.predict(features_flat).reshape(img.shape)

**Alternative:** Predict with saved ONNX model

In [ ]:
import onnxruntime as rt

sess = rt.InferenceSession("rf_hp1_try001.onnx", providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name
pred_onx = sess.run([label_name], {input_name: features_flat.astype(np.float32)})[0]

mask_pred = pred_onx.reshape(img.shape)

In [ ]:
if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.view_image(img)
viewer.add_labels(mask_ref)
viewer.add_labels(mask_pred)